# 05 — Phase 5: Publication-Quality Figures

Produces all five paper figures as high-resolution PNGs and PDFs.

| Fig | Description |
|---|---|
| Fig 1 | Logit-lens curves: conflict vs. unambiguous (per category, ± std band) |
| Fig 2 | Phase transition layer distribution histogram (coloured by category) |
| Fig 3 | Arbitration head heatmap (12×12 activation delta) |
| Fig 4 | Ablation flip rate bar chart (top-5 candidates vs. baseline) |
| Fig 5 | Cross-category overlap heatmap (Jaccard similarity) |

All figures use a consistent colour palette and font sizes appropriate for a
two-column conference paper (NeurIPS / ICLR width = 3.25 in per column).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

P2 = Path('../data/results/phase2')
P3 = Path('../data/results/phase3')
P4 = Path('../data/results/phase4')

# ------------------------------------------------------------------
# Global aesthetics
# ------------------------------------------------------------------
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CAT_COLORS  = {'A': '#4878CF', 'B': '#E47F30', 'C': '#3B9C3A'}
N_LAYERS = 12

print('Setup complete — generating figures...')

## Figure 1 — Logit-Lens Curves

In [ ]:
df_p2   = pd.read_csv(P2 / 'phase2_summary.csv')
layer_x = np.arange(N_LAYERS)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.8), sharey=False)

for ax, cat in zip(axes, ['A', 'B', 'C']):
    color = CAT_COLORS[cat]

    # Load conflict curves
    conf_pids  = df_p2[df_p2.category == cat]['prompt_id'].tolist()
    unamb_pids = [p.replace('_conflict', '_unamb') for p in conf_pids]

    def load_curves(pids, suffix=''):
        curves = []
        for pid in pids:
            f = P2 / f'layer_curves_{pid}.npy'
            if f.exists():
                curves.append(np.load(f))
        return curves

    conf_curves  = load_curves(conf_pids)
    unamb_curves = load_curves(unamb_pids)

    def plot_band(ax, curves, color, label, ls='-'):
        if not curves:
            return
        arr  = np.stack(curves)
        mean = arr.mean(0)
        std  = arr.std(0)
        ax.plot(layer_x, mean, color=color, lw=2, linestyle=ls, label=label, zorder=3)
        ax.fill_between(layer_x, mean-std, mean+std, color=color, alpha=0.18, zorder=2)

    plot_band(ax, conf_curves,  color, f'Conflict (n={len(conf_curves)})')
    plot_band(ax, unamb_curves, color, f'Unambiguous (n={len(unamb_curves)})', ls='--')

    ax.axhline(0, color='#333333', lw=0.9, linestyle=':', zorder=1)

    # Phase transition marker
    tr_vals = [t for t in df_p2[df_p2.category==cat]['phase_transition_layer'] if not pd.isna(t)]
    if tr_vals:
        mean_tr = np.mean(tr_vals)
        ax.axvline(mean_tr, color='firebrick', lw=1.2, linestyle='-.',
                   label=f'Transition L{mean_tr:.1f}', zorder=4)

    cat_labels = {'A': 'Pronoun/Reference', 'B': 'Instruction', 'C': 'Factual Override'}
    ax.set_title(f'Category {cat}: {cat_labels[cat]}')
    ax.set_xlabel('Layer')
    ax.set_xticks(range(0, N_LAYERS, 2))
    ax.legend(loc='upper left', framealpha=0.9)
    ax.grid(alpha=0.25)

axes[0].set_ylabel('Logit diff  (answer A − answer B)')
fig.suptitle('Figure 1: Logit-Lens Curves — Conflict vs. Unambiguous Prompts',
             fontweight='bold', fontsize=11, y=1.02)
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'fig1_logit_lens_curves.{ext}')
plt.show()
print('Figure 1 saved.')

## Figure 2 — Phase Transition Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
bins = np.arange(-0.5, N_LAYERS + 0.5, 1)

bottoms = np.zeros(N_LAYERS)
for cat in ['A', 'B', 'C']:
    vals = [
        int(t) for t in df_p2[df_p2.category == cat]['phase_transition_layer']
        if not pd.isna(t)
    ]
    counts = np.array([vals.count(l) for l in range(N_LAYERS)], dtype=float)
    ax.bar(range(N_LAYERS), counts, bottom=bottoms, label=f'Category {cat}',
           color=CAT_COLORS[cat], edgecolor='white', linewidth=0.6)
    bottoms += counts

ax.set_xlabel('Phase Transition Layer')
ax.set_ylabel('Count')
ax.set_title('Figure 2: Phase Transition Layer Distribution', fontweight='bold')
ax.set_xticks(range(N_LAYERS))
ax.legend(framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'fig2_phase_transition_dist.{ext}')
plt.show()
print('Figure 2 saved.')

## Figure 3 — Arbitration Head Heatmap

In [ ]:
import json
from circuit_conflict.metrics import rank_heads_by_delta

delta   = np.load(P3 / 'activation_delta.npy')  # (12, 12)
ranked  = rank_heads_by_delta(delta)

# Load circuit head lists
P1 = Path('../data/results/phase1')
with open(P1 / 'circuit_A_heads.json') as f:
    circuit_A_heads = set(map(tuple, json.load(f)))
with open(P1 / 'circuit_B_heads.json') as f:
    circuit_B_heads = set(map(tuple, json.load(f)))

fig, ax = plt.subplots(figsize=(7, 5.5))
vmax = np.abs(delta).max()
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im   = ax.imshow(delta, cmap='RdYlGn', norm=norm, aspect='auto', interpolation='nearest')

# Overlay Circuit A heads (blue outline)
for (l, h) in circuit_A_heads:
    ax.add_patch(mpatches.Rectangle((h-0.5, l-0.5), 1, 1,
                  fill=False, edgecolor='royalblue', lw=1.5, linestyle='--'))

# Overlay Circuit B heads (purple outline)
for (l, h) in circuit_B_heads:
    ax.add_patch(mpatches.Rectangle((h-0.5, l-0.5), 1, 1,
                  fill=False, edgecolor='purple', lw=1.5, linestyle='dotted'))

# Label top-5 arbitration candidates with rank numbers
for rank, (l, h, s) in enumerate(ranked[:5], 1):
    ax.add_patch(mpatches.Rectangle((h-0.5, l-0.5), 1, 1,
                  fill=False, edgecolor='black', lw=2.5))
    ax.text(h, l, str(rank), ha='center', va='center',
            fontsize=8, fontweight='bold', color='black')

ax.set_xlabel('Head index')
ax.set_ylabel('Layer')
ax.set_xticks(range(12))
ax.set_yticks(range(12))
ax.set_title('Figure 3: Arbitration Head Candidates (Conflict − Unambiguous Activation Delta)',
             fontweight='bold')

legend_handles = [
    mpatches.Patch(facecolor='none', edgecolor='royalblue', linestyle='--', label='Circuit A head'),
    mpatches.Patch(facecolor='none', edgecolor='purple',    linestyle='dotted', label='Circuit B head'),
    mpatches.Patch(facecolor='none', edgecolor='black',     label='Top-5 arbitration candidate'),
]
ax.legend(handles=legend_handles, loc='upper right', framealpha=0.9, fontsize=8)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Mean activation delta (conflict − unambiguous)')
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'fig3_arbitration_heatmap.{ext}')
plt.show()
print('Figure 3 saved.')

## Figure 4 — Ablation Flip Rate Bar Chart

In [ ]:
df_abl = pd.read_csv(P3 / 'ablation_results.csv')
df_abl = df_abl.sort_values('rank').head(5)

# Compute baseline flip rate (random chance ≈ 50% if model has no signal)
# True baseline: flip rate when ablating a *random* (low-delta) head
df_abl_all = pd.read_csv(P3 / 'ablation_results.csv').sort_values('delta_score')
baseline_flip_rate = df_abl_all.head(5)['flip_rate'].mean()  # bottom-5 by delta

fig, ax = plt.subplots(figsize=(7, 4.2))

labels = [f'L{int(r.layer)}H{int(r.head)}\n(rank {int(r.rank)})' for _, r in df_abl.iterrows()]
flip_rates = df_abl['flip_rate'].values

bars = ax.bar(labels, flip_rates * 100, color='#E47F30', edgecolor='black',
              linewidth=0.7, width=0.55, label='Top-5 candidate heads')
ax.axhline(baseline_flip_rate * 100, color='steelblue', lw=2,
           linestyle='--', label=f'Baseline (bottom-5 heads, {baseline_flip_rate:.1%})')

# Annotate bars
for bar, rate in zip(bars, flip_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, max(flip_rates.max() * 110, baseline_flip_rate * 130))
ax.set_ylabel('Ablation flip rate (%)')
ax.set_title('Figure 4: Ablation Flip Rate — Top-5 Arbitration Head Candidates',
             fontweight='bold')
ax.legend(framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'fig4_ablation_flip_rates.{ext}')
plt.show()
print('Figure 4 saved.')

## Figure 5 — Cross-Category Overlap (Jaccard Heatmap)

In [ ]:
jaccard_df = pd.read_csv(P3 / 'jaccard_matrix.csv', index_col=0)

fig, ax = plt.subplots(figsize=(4.5, 3.8))
cats = list(jaccard_df.index)
mat  = jaccard_df.values.astype(float)

im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')

# Annotate cells with Jaccard score
for i in range(len(cats)):
    for j in range(len(cats)):
        text_color = 'white' if mat[i, j] > 0.6 else 'black'
        ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color=text_color)

ax.set_xticks(range(len(cats)))
ax.set_yticks(range(len(cats)))
ax.set_xticklabels([f'Cat {c}' for c in cats])
ax.set_yticklabels([f'Cat {c}' for c in cats])
ax.set_title('Figure 5: Cross-Category Jaccard Similarity\n(Top-10 Arbitration Heads)',
             fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Jaccard similarity')
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'fig5_cross_category_jaccard.{ext}')
plt.show()
print('Figure 5 saved.')

## Supplementary: Circuit A & B maps (one per category)

In [ ]:
P1 = Path('../data/results/phase1')

fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), sharey=True)
for ax, cat in zip(axes, ['A', 'B', 'C']):
    f = P1 / f'circuit_map_cat{cat}.npy'
    if not f.exists():
        ax.set_title(f'Cat {cat}: no data')
        continue
    contrib = np.load(f)
    vmax = np.abs(contrib).max()
    im = ax.imshow(contrib, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    ax.set_title(f'Category {cat}')
    ax.set_xlabel('Head')
    ax.set_xticks(range(12))
    ax.set_yticks(range(12))
    plt.colorbar(im, ax=ax, label='Causal Δ logit_diff')

axes[0].set_ylabel('Layer')
fig.suptitle('Supplementary: Baseline Circuit Maps (Phase 1)',
             fontweight='bold', y=1.02)
plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(FIGURES_DIR / f'figS1_baseline_circuit_maps.{ext}')
plt.show()
print('Supplementary figure S1 saved.')
print('\nPhase 5 complete — all figures saved to figures/ ✓')